# Cattle breed classification: reproducible thesis workflow

**15 classes · ImageNet-pretrained ResNet-18 · PyTorch**

This notebook brings the existing `dataset_loader.py`, `train.py`, `evaluate.py`
and `predict.py` workflows into one ordered experiment. It imports the tested
implementation rather than maintaining a second, potentially inconsistent model.
Keep the notebook in the project directory with those files.

**Two distinct forms of evidence:** dataset-analysis figures can be generated now
from the measured integrity audit. Model-performance figures require a completed
non-smoke run and its matching held-out evaluation. Smoke-test scores never enter
the thesis-performance exports. No accuracy is invented or interpolated.

Figures export as **300-DPI PNG** and **editable SVG**, with captions, numeric CSV
tables and provenance manifests. Cite the source dataset when reproducing its
images; the analysis and experiment figures are generated by this workflow.


## 1. Open and configure

From PowerShell in this directory:
```powershell
py -m pip install -r requirements-notebook.txt
py -m jupyterlab Cattle_Breed_Classification_Thesis.ipynb
```
Select the Python environment containing PyTorch. Run cells from top to bottom.
The default configuration renders the existing audit and available results without
starting another long CPU training job. To train from here, choose a **new run
directory**, set `RUN_TRAINING=True`, and run the training cell. After completing
validation-based model selection, set `RUN_EVALUATION=True` and run evaluation once.


In [ ]:
from pathlib import Path
import sys, json, subprocess, hashlib, inspect
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown, Image as DisplayImage, Code

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "train.py").is_file():
    raise RuntimeError("Open this notebook from the project folder containing train.py.")
sys.path.insert(0, str(PROJECT_ROOT))
from dataset_loader import (resolve_root, audit_dataset, apply_duplicate_policy, require_clean,
                            load_rgb, build_transform, preprocessing_config)
from pipeline_utils import save_json, environment, seed_everything
from thesis_figures import (FigureWriter, dataset_figures, training_figures,
                            evaluation_figures, completed_run)

DATA_ROOT = resolve_root(PROJECT_ROOT / "Master-Dataset/master_dataset")
AUDIT_PATH = PROJECT_ROOT / "dataset_audit.json"
RUN_DIR = PROJECT_ROOT / "runs/resnet18_thesis"
EVALUATION_DIR = PROJECT_ROOT / "evaluation/resnet18_thesis"
FIGURE_ROOT = PROJECT_ROOT / "thesis_outputs"
NEW_IMAGE_PATH = None  # Set to Path(r"C:/photos/my_cattle.jpg") for a new prediction.

REFRESH_AUDIT = False
RUN_TRAINING = False
RUN_EVALUATION = True
EPOCHS, BATCH_SIZE, IMAGE_SIZE = 15, 16, 224
LEARNING_RATE, WEIGHT_DECAY = 0.001, 0.0001
SEED, WORKERS, THREADS = 42, 0, 4
PATIENCE, SCHEDULER_PATIENCE, MIN_DELTA = 5, 2, 0.0001
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FINE_TUNE = False  # False: frozen backbone + trainable 15-logit head.
DUPLICATE_POLICY = "exclude-all"

seed_everything(SEED, THREADS)
print("Python:", sys.executable)
print("Dataset:", DATA_ROOT)
print("Device:", DEVICE)
display(pd.DataFrame(environment()["packages"].items(), columns=["Package", "Version"]))


## 2. Audit and dataset definition

The actual archive contains an extra `master_dataset` directory. The loader
recognizes it, discovers the 15 class names from **training folders only**, and
uses the same mapping for validation, test and inference. Legacy manifests with
the original author's absolute paths are not used for image loading.

The recorded audit decoded 14,443 files and found 15 cross-split duplicate pairs
and 30 within-split conflicting-label groups. The explicit policy excludes every
copy of those conflicts without deleting source images. Exact byte/pixel hashing
does not certify the absence of perceptually similar or augmented near-duplicates.

`REFRESH_AUDIT=False` plots the **saved audit snapshot**, avoiding another full
scan. Set it to `True` after changing any data. Full training always audits afresh.
Test images are inspected for integrity only; they do not enter training/tuning.


In [ ]:
if REFRESH_AUDIT or not AUDIT_PATH.is_file():
    audit = apply_duplicate_policy(audit_dataset(DATA_ROOT), DUPLICATE_POLICY)
    save_json(AUDIT_PATH, audit)
else:
    audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
require_clean(audit)
classes = sorted(audit["class_to_idx"], key=audit["class_to_idx"].get)
assert len(classes) == 15
counts = pd.DataFrame({
    **{f"raw_{s}": [audit["counts"][s][c] for c in classes] for s in ("train", "val", "test")},
    **{f"used_{s}": [audit["effective_counts"][s][c] for c in classes] for s in ("train", "val", "test")},
}, index=classes)
counts.index.name = "breed"
display(counts)
display(counts.sum().to_frame("total_images"))
print("Explicit exclusions:", len(audit["exclusions"]))
print("Audit fingerprint:", audit["dataset_fingerprint"])
display(pd.DataFrame(audit["exclusions"])[["split", "path", "reason"]])


## 3. Dataset charts and representative training images

These figures describe the measured dataset, not classifier accuracy. Counts and
class weights are calculated from the audit, not hardcoded breed assumptions.
The gallery selects one retained training image per label by file order; labels
are dataset annotations, not independently verified breed judgments.


In [ ]:
def show_figures(paths, width=900):
    for path in paths:
        manifest = json.loads((path.parent / "figure_manifest.json").read_text(encoding="utf-8"))
        entry = next(row for row in manifest["figures"] if row["png"] == path.name)
        display(DisplayImage(filename=str(path), width=width))
        display(Markdown("**Suggested caption:** " + entry["caption"]))

dataset_paths = dataset_figures(AUDIT_PATH, FIGURE_ROOT / "dataset", DATA_ROOT)
show_figures(dataset_paths)
print("Numeric counts:", FIGURE_ROOT / "dataset/dataset_counts.csv")


## 4. Preprocessing and augmentation

ResNet-18 uses ImageNet mean `[0.485, 0.456, 0.406]` and standard deviation
`[0.229, 0.224, 0.225]`. Validation/test/inference resize the shorter side to 256
then center-crop 224×224 at the default resolution. Training alone adds random
resized cropping, horizontal flipping and mild color jitter. RGB conversion, EXIF
orientation and white-background alpha compositing are shared across all paths.

The following figure transforms **one training image repeatedly**. It demonstrates
augmentation; it is not evidence of increased independent sample size.


In [ ]:
import matplotlib.pyplot as plt
preprocessing = preprocessing_config(IMAGE_SIZE)
display(preprocessing)
excluded = {r["path"] for r in audit["exclusions"]}
example = next(r for r in audit["files"] if r["split"] == "train" and r["path"] not in excluded)
original = load_rgb(DATA_ROOT / example["path"])
transform = build_transform(preprocessing, training=True)
seed_everything(SEED, THREADS)
fig, axes = plt.subplots(1, 5, figsize=(14, 3.5), layout="constrained")
axes[0].imshow(original)
axes[0].set_title("Original training image")
for i, ax in enumerate(axes[1:], 1):
    tensor = transform(original)
    rgb = tensor.permute(1, 2, 0).numpy() * np.array(preprocessing["std"]) + np.array(preprocessing["mean"])
    ax.imshow(np.clip(rgb, 0, 1))
    ax.set_title(f"Random transform {i}")
for ax in axes:
    ax.axis("off")
augmentation_writer = FigureWriter(FIGURE_ROOT / "augmentation", {
    "kind": "method_illustration", "source_image": example["path"], "seed": SEED,
    "preprocessing": preprocessing, "dataset_fingerprint": audit["dataset_fingerprint"]})
augmentation_path = augmentation_writer.save(fig, "training_augmentation", 
    "One retained training photograph and four seeded random training transformations. "
    "These are views of the same image, not additional independent observations. Cite the source dataset.")
show_figures([augmentation_path], width=1100)


## 5. Model and training method

The runnable training implementation comes from `train.py`. It initializes
`ResNet18_Weights.IMAGENET1K_V1`, replaces the final layer with 15 logits, and by
default trains only that layer while freezing the backbone and batch-normalization
statistics. `FINE_TUNE=True` trains the entire network; use validation to select
its learning rate and do not use test results to tune it.

After exclusions the training class-count ratio exceeds 2, so automatic class
weighting uses `N_train / (15 * class_count)`. Only training loss is weighted.
Validation/test metrics are unweighted. The best checkpoint maximizes validation
macro F1. Early stopping, AdamW and ReduceLROnPlateau use validation only.


In [ ]:
from train import run_epoch
display(pd.DataFrame([{
    "architecture": "ResNet-18", "pretraining": "ImageNet-1K V1", "classes": 15,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY, "image_size": IMAGE_SIZE, "seed": SEED,
    "fine_tune": FINE_TUNE, "selection": "validation macro F1",
    "early_stop_patience": PATIENCE, "scheduler_patience": SCHEDULER_PATIENCE,
}]).T.rename(columns={0: "configured value"}))
# Display the exact shared training/validation loop; no second implementation.
display(Code(inspect.getsource(run_epoch), language="python"))


## 6. Run full training, or reuse an existing run

Set `RUN_TRAINING=True` in the configuration to launch training here. The notebook
uses the same Python kernel's executable and streams epoch logs. It refuses to
overwrite an existing output directory. If training was already started in
PowerShell, leave the flag false and wait for that run to finish.

Windows process handling stays inside the `.py` entry point, so configurable data
loader workers work without copying multiprocessing code into notebook cells.


In [ ]:
def run_pipeline(script, arguments):
    command = [sys.executable, "-u", str(PROJECT_ROOT / script), *map(str, arguments)]
    print("Command:", subprocess.list2cmdline(command), flush=True)
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait() != 0:
            raise RuntimeError(f"{script} failed with exit code {process.returncode}; see the log above.")
    except BaseException:
        if process.poll() is None:
            process.terminate()
            process.wait()
        raise

if RUN_TRAINING:
    if RUN_DIR.exists():
        raise RuntimeError("RUN_DIR already exists. Reuse its results or choose a new run name; do not overwrite it.")
    arguments = ["--data-root", DATA_ROOT, "--output", RUN_DIR, "--duplicates", DUPLICATE_POLICY,
                 "--epochs", EPOCHS, "--batch-size", BATCH_SIZE, "--lr", LEARNING_RATE,
                 "--weight-decay", WEIGHT_DECAY, "--image-size", IMAGE_SIZE, "--seed", SEED,
                 "--workers", WORKERS, "--threads", THREADS, "--device", DEVICE,
                 "--patience", PATIENCE, "--scheduler-patience", SCHEDULER_PATIENCE, "--min-delta", MIN_DELTA]
    if FINE_TUNE:
        arguments.append("--fine-tune")
    run_pipeline("train.py", arguments)
else:
    print("Training launch disabled. Reading existing results when complete:", RUN_DIR)


## 7. Learning curves and selected epoch

These plots are generated only from a completed non-smoke checkpoint with matching
history. The dashed line marks the selected epoch. Loss is shown with a caveat:
weighted training loss and unweighted validation loss have different aggregation
weights. Accuracy and macro F1 use fixed 0–1 axes; values are not smoothed.


In [ ]:
training_ready = False
try:
    checkpoint, history = completed_run(RUN_DIR)
except (OSError, ValueError) as error:
    display(Markdown(f"**Training results pending:** {error}"))
else:
    training_ready = True
    display(pd.DataFrame(history))
    display(Markdown(f"Selected epoch: **{checkpoint['best_epoch']}**. "
                     f"Selection metric: **{checkpoint['selection_metric']}**."))
    display(checkpoint["training_config"])
    training_paths = training_figures(RUN_DIR, FIGURE_ROOT / RUN_DIR.name / "training")
    show_figures(training_paths)


## 8. Final held-out test evaluation

Enable `RUN_EVALUATION` only after finishing training and validation-based model
selection. Evaluation loads the original checkpoint mapping/preprocessing and
verifies that the test inventory has not changed. It applies saved exclusions and
rejects overlap with images used for training or selection.

The pipeline reports accuracy, per-breed precision/recall/F1/support, macro and
weighted averages. Undefined values use `zero_division=0`. Existing evaluation
outputs are reused only when their recorded checkpoint hash matches.


In [ ]:
if RUN_EVALUATION:
    if not training_ready:
        raise RuntimeError("Finish full training and select a checkpoint before test evaluation.")
    if not EVALUATION_DIR.exists():
        run_pipeline("evaluate.py", ["--checkpoint", RUN_DIR / "best_model.pt", "--data-root", DATA_ROOT,
                      "--output", EVALUATION_DIR, "--device", DEVICE, "--batch-size", BATCH_SIZE,
                      "--workers", WORKERS, "--threads", THREADS])
    else:
        print("Evaluation directory exists; validating and reading its saved outputs below.")
else:
    print("Test evaluation launch disabled; no new test predictions are generated.")


## 9. Thesis result figures and numeric tables

Available only after a real evaluation: count and row-normalized confusion
matrices, per-breed precision/recall/F1, aggregate accuracy/macro-F1/weighted-F1,
and the ten most frequent directed confusions. Captions describe measurements,
not unverified causes. Error counts do not explain why breeds were confused.

The exporter checks checkpoint identity and recomputes metrics from saved
predictions before drawing charts. No ROC/PR curves are fabricated from the
top-class confidence alone. Single-run scores are not cross-validation averages.


In [ ]:
evaluation_ready = False
if training_ready and (EVALUATION_DIR / "metrics.json").is_file():
    result_paths = evaluation_figures(RUN_DIR, EVALUATION_DIR, FIGURE_ROOT / RUN_DIR.name / "test")
    evaluation_ready = True
    show_figures(result_paths)
    display(pd.read_csv(EVALUATION_DIR / "classification_report.csv"))
else:
    display(Markdown("**Final test figures pending.** Complete training and evaluation; then rerun this section."))


## 10. Predict one new image

Set `NEW_IMAGE_PATH` to a real image path in the configuration. The reusable
`BreedPredictor` loads once, applies checkpoint preprocessing and returns the top
three classes. The probability plot is one-image model output, not test accuracy.
Softmax confidence is uncalibrated, and this closed-set classifier always chooses
one of its 15 labels, including for unrelated inputs.


In [ ]:
if NEW_IMAGE_PATH is not None and training_ready:
    from predict import BreedPredictor
    predictor = BreedPredictor(RUN_DIR / "best_model.pt", DEVICE)
    prediction = predictor.predict(NEW_IMAGE_PATH)
    display(prediction)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    axes[0].imshow(load_rgb(NEW_IMAGE_PATH))
    axes[0].axis("off")
    axes[0].set_title("New input image")
    top = prediction["top_three"]
    axes[1].barh([r["breed"] for r in top], [r["confidence"] for r in top], color=["#0072B2", "#E69F00", "#009E73"])
    axes[1].invert_yaxis()
    axes[1].set(xlim=(0, 1), xlabel="Softmax confidence", title="Top three predictions")
    prediction_writer = FigureWriter(FIGURE_ROOT / RUN_DIR.name / "prediction", {
        "kind": "single_image_example", "image": str(NEW_IMAGE_PATH),
        "checkpoint_sha256": hashlib.sha256((RUN_DIR / "best_model.pt").read_bytes()).hexdigest()})
    prediction_path = prediction_writer.save(fig, "single_image_prediction",
        "Prediction for one new image. Bars show uncalibrated softmax scores, not overall classifier accuracy.")
    save_json(prediction_writer.directory / "prediction.json", prediction)
    show_figures([prediction_path])
else:
    print("Prediction pending: choose NEW_IMAGE_PATH and finish training first.")


## 11. Export and thesis reporting checklist

All figures are saved under `thesis_outputs/`. Use PNG in Word; SVG preserves
editable vector text/lines. Each section has `captions.md`, `figure_manifest.json`
and supporting numeric files. Keep these with the checkpoint/config/history as
experiment evidence. If results are rerun, regenerate figures from that same run.

Report the actual exclusions, split sizes, weighting method, frozen/fine-tuned
backbone, seed, software versions, selected epoch and selection metric. Report
performance only after full training and held-out evaluation. Do not describe
dataset exploration or smoke tests as final classification results.

**Sources:** [dataset](https://www.kaggle.com/datasets/vedant0dusane/cattle-breed-classification-dataset),
[ResNet-18 weights and preprocessing](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html).
Preserve `LICENSE` and `CITATION.cff`; follow your institution's disclosure rules
for tools used to implement the experiment.


In [ ]:
manifest_paths = sorted(FIGURE_ROOT.rglob("figure_manifest.json"))
export_rows = []
for path in manifest_paths:
    manifest = json.loads(path.read_text(encoding="utf-8"))
    for item in manifest["figures"]:
        export_rows.append({"section": path.parent.relative_to(FIGURE_ROOT).as_posix(),
                            "figure": item["name"], "PNG": str(path.parent / item["png"]),
                            "SVG": str(path.parent / item["svg"])})
display(pd.DataFrame(export_rows))
print("Completed non-smoke training available:", training_ready)
print("Matching held-out evaluation available:", evaluation_ready)
print("Dataset charts and model results are separate evidence categories.")
